In [ ]:
# NOTEBOOK NAME
# PPIvarianceSaver.ipynb
# NOTEBOOK NAME

# # OPENING IMPORTS
import numpy as np
from matplotlib import pyplot as plt
import xarray as xr

from pathlib import Path      # used to play with pathnames to save

# # for projecting radar coordinates to lat and lon
# from pyproj import Geod

# # mapping things
# import cartopy.crs as ccrs
# import cartopy.feature as cfeature
# from cartopy.io import shapereader

# # for adding lat/lon gridlines on plots
# import matplotlib.ticker as mticker
# from cartopy.mpl.gridliner import LATITUDE_FORMATTER, LONGITUDE_FORMATTER

# # SPECIAL METHOD TO IMPORT CUSTOM FUNCTIONS AND ELEVATION FROM LOCAL DIRECTORY
import sys
sys.path.append('/home/563/sg3241/Notebooks/PhD/CustomFunctions')
from CustomFunctions1 import *

# # for adding a colourful topo base map to the CAPI plots
# from custom_elevation import fetch_srtm, fetch_gebco_local
# from matplotlib.colors import LinearSegmentedColormap
# from matplotlib.colors import ListedColormap, BoundaryNorm, Normalize

In [ ]:
# ADD VARIANCE VARIABLES TO THE EXISTING PPI DATA

# CHOOSE THE RADAR
RadarIDno = '19'  # 22 is Mackay, 106 is Townsville, 66 is Mt Staplyton (Brisbane) , 50 is Marabong (near Brisbane)

# CHOOSE THE DATE
# the day in consideration (YYYYMMDD) and time (hhmmss)       ALL IN UTC !!!
RadarYear  = 2024
RadarMonth = 1
RadarDay   = 1
# date choice follow-on
# add leading zeros for strings
YYYY = str(RadarYear).zfill(4)
MM = str(RadarMonth).zfill(2)
DD = str(RadarDay).zfill(2)
# write out the data in one string with and without dashes
RadarFileDate  = YYYY + MM + DD
RadarFileDatePrint = YYYY + '-' + MM + '-' + DD

# CHOOSE THE MINUTES YOU WANT TO LOOP OVER (5-MIN PERIODS) (INCLUSIVE OF START AND END TIMES)
LoopStartTime = '00:00'
LoopEndTime   = '23:55'

# LOOP OVER EVERY 5 MIN PERIOD IN THE DAY
# Parse start and end times
StartHour, StartMin = int(LoopStartTime.split(':')[0]), int(LoopStartTime.split(':')[1])
EndHour, EndMin     =   int(LoopEndTime.split(':')[0]),   int(LoopEndTime.split(':')[1])

# Convert to total minutes for easy comparison
StartMinOfDay = StartHour * 60 + StartMin
EndMinOfDay = EndHour * 60 + EndMin


for MinOfDay in range(StartMinOfDay, EndMinOfDay + 1, 5):
    # find the old indicies with which this code was written (0-23 for hours, 0-11 for 5-minute periods within hours)
    houri = MinOfDay // 60
    mini  = MinOfDay % 60
        
    RadarFileTime = str(houri).zfill(2) + str(mini).zfill(2) + '00' # write out the time in 6 digits (like 012040 for 01:20:40 AM)
    # add a string of format hh:mm:ss for printing
    RadarFileTimePrint = str(RadarFileTime)[0:2] + ':' + str(RadarFileTime)[2:4] + ':' + str(RadarFileTime)[4:6] 
    
    print('working on ' + RadarFileTimePrint)

    
    NetCDFstorageFolder = '/scratch/v46/sg3241/tmp/UnzippedRadarFiles/QC0/Radar' + RadarIDno + '/' + YYYY + '/' + RadarIDno + '_' + RadarFileDate + '_ppi' + '/'
    NetCDFstorageFile = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi.nc'
    NetCDFstoragePath = NetCDFstorageFolder + NetCDFstorageFile
    
    
    # try to load in the netcdf file and if it doesn't work, just keep going
    try:
        RadarXR = xr.open_dataset(NetCDFstoragePath, decode_timedelta = False) # add the decode_timedelta to shut up a warning
    except FileNotFoundError:
        print(f'File missing for {RadarFileTimePrint}, skipping: {NetCDFstoragePath}')
        continue
    
    
    GridStats(RadarXR, 'corrected_reflectivity', 3, fill_value=-32.0)                                   # add variances and counts for Z
        
    SaveFolder = NetCDFstorageFolder
    SaveFile = RadarIDno + '_' + RadarFileDate + '_' + RadarFileTime + '_ppi_with_variances.nc'
    SavePath = SaveFolder + SaveFile

    # CHAD EDITTED SAVING SECTION WHICH DELETES AND RE-WRITES SAVED FILES
    # Ensure directory exists with proper permissions
    if not Path(SaveFolder).exists():
        Path(SaveFolder).mkdir(parents=True, exist_ok=True)
    
    # Remove existing file to avoid permission conflicts
    if Path(SavePath).exists():
        Path(SavePath).unlink()

    # CHAD EDDITED SAVING BLOCK TO KEEP THE SAVED FORMAT AS INT FOR VARIABLES AND MAKE THE FILES ~12% the size
    # You need to preserve the original encoding for existing variables and specify encoding for the new ones. Here is a corrected saving block:
    
        # --- Build encoding dict ---

    # Step 1: Capture original encoding for pre-existing variables,
    #         but strip read-only keys that netCDF4 will reject
    STRIP_KEYS = {'source', 'original_shape', 'preferred_chunks', 'szip', 'zstd', 'bzip2', 'blosc', 'char_dim_name', 'coordinates'}

    original_encoding = {}
    for var in RadarXR.data_vars:
        enc = {k: v for k, v in RadarXR[var].encoding.items() if k not in STRIP_KEYS}
        if enc:
            original_encoding[var] = enc

    # Step 2: Define encoding for the new variance/count variables
    new_var_names = [
        'corrected_reflectivity_3x3grid_variance',
        'corrected_reflectivity_3x3grid_count',
        'corrected_cross_correlation_ratio_3x3grid_variance',
        'corrected_cross_correlation_ratio_3x3grid_count',
    ]

    # Compute int16 packing parameters for variance variables
    def compute_int16_packing(data_min, data_max):
        """Compute scale_factor and add_offset for int16 packing."""
        scale_factor = (data_max - data_min) / (2**16 - 2)  # leave 1 value for _FillValue
        add_offset   = data_min + 2**15 * scale_factor
        return scale_factor, add_offset

    # Define ranges for each variance variable
    variance_ranges = {
        'corrected_reflectivity_3x3grid_variance':            (0.0, 1600.0),
        'corrected_cross_correlation_ratio_3x3grid_variance': (0.0, 0.25),
    }

    new_var_encoding = {}
    for name in new_var_names:
        if name not in RadarXR.data_vars:
            continue

        if name.endswith('_count'):
            # int8 for count (values 0-9 for a 3x3 grid)
            new_var_encoding[name] = {
                'dtype':      'int8',
                'zlib':       True,
                'complevel':  4,
                'shuffle':    True,
                '_FillValue': np.int8(-1),
            }

        elif name.endswith('_variance'):
            # int16 with scale/offset packing for variance
            vmin, vmax = variance_ranges[name]
            sf, ao = compute_int16_packing(vmin, vmax)
            new_var_encoding[name] = {
                'dtype':        'int16',
                'zlib':         True,
                'complevel':    4,
                'shuffle':      True,
                '_FillValue':   np.int16(-32768),
                'scale_factor': sf,
                'add_offset':   ao,
            }

    # Step 3: Merge, letting original encoding take priority
    save_encoding = {**new_var_encoding, **original_encoding}

    # --- Write with explicit mode ---
    try:
        RadarXR.to_netcdf(
            SavePath,
            mode='w',
            encoding=save_encoding
        )
        print(f"Successfully saved: {SavePath}")
    except PermissionError as e:
        print(f"Permission denied writing to {SavePath}: {e}")
